# Numbers all the way down

This notebook is about a big idea behind language models: **inside the model, text becomes numbers**.

In Eric's earlier notebook, this story was told with local GGUF files and Python tools that can inspect model internals directly. In the browser, we have a different set of tools, so this version focuses on the parts we can still see clearly with WebLLM: token counts, chunking, and embeddings.

An **embedding** is a list of numbers that tries to capture the meaning of a piece of text.

## Step 1: Get set up

This loads WebLLM, checks for WebGPU, and defines helpers for loading a chat model and an embedding model.

In [ ]:
import js
import math
import pandas as pd
from pyodide.ffi import to_js, create_proxy
from IPython.display import display

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

chat_engine = None
embed_engine = None
loaded_chat_model = None
loaded_embed_model = None


def js_obj(d):
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def load_chat_model(model_id):
    global chat_engine
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        global chat_engine
        if chat_engine is None:
            if chat_options:
                chat_engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}), js_obj(chat_options)
                )
            else:
                chat_engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress})
                )
        else:
            chat_engine.setInitProgressCallback(progress)
            if chat_options:
                await chat_engine.reload(model_id, js_obj(chat_options))
            else:
                await chat_engine.reload(model_id)

    try:
        await attempt()
    except Exception as error:
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise


async def load_embedding_model(model_id):
    global embed_engine
    status, progress = progress_bar()
    if embed_engine is None:
        embed_engine = await webllm.CreateMLCEngine(
            model_id, js_obj({"initProgressCallback": progress})
        )
    else:
        embed_engine.setInitProgressCallback(progress)
        await embed_engine.reload(model_id)


async def chat_usage(user_text, max_tokens=1):
    if loaded_chat_model is None:
        return {"error": "No chat model is loaded yet. Run Step 3."}
    request = {
        "messages": [{"role": "user", "content": user_text}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "frequency_penalty": 0.5,
    }
    reply = await chat_engine.chat.completions.create(js_obj(request))
    return reply.to_py()


async def embed_texts(texts):
    if loaded_embed_model is None:
        return {"error": "No embedding model is loaded yet. Run Step 9."}
    reply = await embed_engine.embeddings.create({"input": texts})
    return [item.embedding for item in reply.data]


## Step 2: Choose your models

We use one small chat model and one embedding model.

In [ ]:
CHAT_MODEL_ID = "gemma3-1b-it-q4f16_1-MLC"
EMBED_MODEL_ID = "snowflake-arctic-embed-m-q0f32-MLC-b4"

print("Chat model:      " + CHAT_MODEL_ID)
print("Embedding model: " + EMBED_MODEL_ID)

## Step 3: Load the chat model

If the model is not saved in your browser yet, this will download it first.

In [ ]:
await load_chat_model(CHAT_MODEL_ID)
loaded_chat_model = CHAT_MODEL_ID
print(loaded_chat_model + " is loaded and ready.")

## Step 4: Raw text is still just text to us

Before the model sees text as tokens, we can still measure easy things like characters and bytes. A **byte** is a very small storage unit. Computers store text as bytes before higher-level tools turn it into something else.

In [ ]:
TEXT = "A tariff is a tax on imported goods that raises their price."

print(TEXT)
print()
print("Characters:", len(TEXT))
print("Bytes in UTF-8:", len(TEXT.encode('utf-8')))

## Step 5: Prompt tokens are the model's input units

A **token** is a chunk of text the model reads as one unit. Tokens are not always words. Some are short word pieces, punctuation marks, or even spaces attached to words.

WebLLM does not directly show us the tokenizer pieces in this notebook, but it does report how many prompt tokens it used.

In [ ]:
examples = [
    "trade",
    "international trade",
    "International trade can raise living standards.",
    "International trade can raise living standards, but it can also create winners and losers.",
]

rows = []
for text in examples:
    reply = await chat_usage(text, max_tokens=1)
    if 'error' in reply:
        print(reply['error'])
        break
    usage = reply.get('usage', {})
    rows.append({
        'text': text,
        'characters': len(text),
        'prompt_tokens': usage.get('prompt_tokens'),
    })

pd.DataFrame(rows)

## Step 6: Small wording changes can change token counts

Even when two sentences mean almost the same thing, their token counts may differ.

In [ ]:
comparisons = [
    "inflation hurts savers",
    "Inflation hurts savers.",
    "Inflation hurts some savers.",
    "Inflation hurts some savers, especially if prices rise quickly.",
]

rows = []
for text in comparisons:
    reply = await chat_usage(text, max_tokens=1)
    if 'error' in reply:
        print(reply['error'])
        break
    usage = reply.get('usage', {})
    rows.append({
        'text': text,
        'prompt_tokens': usage.get('prompt_tokens'),
    })

pd.DataFrame(rows)

## Step 7: Output is also counted in tokens

The model writes its answer token by token. Here we ask the same question with different output limits.

In [ ]:
PROMPT = "Explain comparative advantage in one short paragraph."

rows = []
for limit in [20, 60, 120]:
    reply = await chat_usage(PROMPT, max_tokens=limit)
    if 'error' in reply:
        print(reply['error'])
        break
    usage = reply.get('usage', {})
    text = reply['choices'][0]['message']['content']
    rows.append({
        'max_tokens': limit,
        'finish_reason': reply['choices'][0].get('finish_reason'),
        'prompt_tokens': usage.get('prompt_tokens'),
        'completion_tokens': usage.get('completion_tokens'),
        'preview': text[:80] + ('...' if len(text) > 80 else ''),
    })

pd.DataFrame(rows)

## Step 8: Long text gets broken into chunks for many tasks

A **chunk** is a smaller piece cut from a longer text. Chunking matters for search and memory systems because you often store or compare one chunk at a time.

Here we make overlapping chunks by character length, then measure their prompt token counts.

In [ ]:
PASSAGE = (
    "Comparative advantage says that people or countries should focus on what they can produce at a lower opportunity cost. "
    "Even if one side is better at making everything, trade can still help both sides if they specialize differently. "
    "That is why economists often use comparative advantage to explain the gains from trade."
)


def chunk_text(text, chunk_size=120, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

chunks = chunk_text(PASSAGE, chunk_size=120, overlap=30)
rows = []
for i, chunk in enumerate(chunks, start=1):
    reply = await chat_usage(chunk, max_tokens=1)
    if 'error' in reply:
        print(reply['error'])
        break
    usage = reply.get('usage', {})
    rows.append({
        'chunk': i,
        'characters': len(chunk),
        'prompt_tokens': usage.get('prompt_tokens'),
        'text': chunk,
    })

pd.DataFrame(rows)

## Step 9: Load the embedding model

Now we switch from counting tokens to looking at embeddings, which are also numbers.

In [ ]:
await load_embedding_model(EMBED_MODEL_ID)
loaded_embed_model = EMBED_MODEL_ID
print(loaded_embed_model + " is loaded and ready.")

## Step 10: Embeddings are vectors of numbers

A **vector** is an ordered list of numbers. The embedding model turns each input into one vector.

In [ ]:
words = [
    "inflation",
    "deflation",
    "tariff",
    "trade",
    "recession",
]

formatted_words = ["[CLS] " + word + " [SEP]" for word in words]
embeddings = await embed_texts(formatted_words)
if isinstance(embeddings, dict) and 'error' in embeddings:
    print(embeddings['error'])
else:
    first_vector = embeddings[0]
    print("The first 12 numbers for 'inflation':")
    print(first_vector[:12])
    print()
    print("Vector length:", len(first_vector))

## Step 11: Similar meanings should have more similar vectors

We can compare two vectors with **cosine similarity**. A higher score means the vectors point in more similar directions.

In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


if isinstance(embeddings, dict) and 'error' in embeddings:
    print(embeddings['error'])
else:
    pairs = []
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            pairs.append({
                'word_1': words[i],
                'word_2': words[j],
                'similarity': cosine_similarity(embeddings[i], embeddings[j]),
            })
    similarity_table = pd.DataFrame(pairs).sort_values('similarity', ascending=False).reset_index(drop=True)
    similarity_table

## Step 12: Putting it all together

When you type a prompt, the browser-side model does not think in whole sentences. Very roughly, the path looks like this:

1. raw text
2. token chunks and token counts
3. many layers of number-crunching inside the model
4. new output tokens
5. decoded text again

For embeddings, the path ends in a vector of numbers instead of a written answer.

## Your turn

Try your own words in the token-count and embedding cells. Which pairs look most similar? Which sentences get more tokens than you expected?